# 🚀 Data Science Capstone: E-Commerce Customer Churn Prediction
**Author:** Daniel Roger Kennedy  
**Program:** Information Systems Business (Semester 4), Universitas Ciputra  
**Methodology:** CRISP-DM  

---

## Phase 1: Business Understanding

### 1. Problem Statement
The e-commerce platform currently relies on a reactive, heuristic-based retention strategy (e.g., blanket discounts to inactive users). This results in high **Customer Acquisition Cost (CAC)** waste and unmitigated revenue attrition from high-value customers who churn quietly.

### 2. Business & Machine Learning Objectives
* **Business Objective:** Identify behavioral churn triggers to proactively allocate promotional budgets to high-risk customer cohorts, thereby reducing overall revenue attrition and optimizing marketing ROI.
* **ML Objective:** Build a classification model to predict the `Churn` flag. We will optimize for **Recall** to identify as many true flight-risk customers as possible, while monitoring Precision to avoid excessive budget waste on false positives.

### 3. Baseline Strategy vs. ML Advantage
* **Current Baseline:** Naive retention heuristic (e.g., sending promos to all users inactive for 30+ days).
* **ML Advantage:** Utilizing multivariate behavioral analysis (`Tenure`, `Complaints`, `App Usage`, `Cashback trends`) to target *only* true flight-risks before they leave the platform.


In [ ]:
# Phase 2: Data Understanding - Initialization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style for professional presentation
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully. Ready for EDA.")

In [ ]:
# Load the dataset
df = pd.read_csv('EComm.csv')

# Display the first 5 rows to verify
df.head()

In [ ]:
# 1. Dataset Shape
print(f"Dataset contains {df.shape[0]} rows and {df.shape[1]} columns.\n")

# 2. Quick breakdown of top missing values
data_info = pd.DataFrame({
    'Data Type': df.dtypes,
    'Missing Values': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df)) * 100
})

# Sort to show columns with highest missing values first
data_info.sort_values(by='Missing %', ascending=False).head(10)

In [ ]:
# Analyzing the Target Variable: Churn
churn_counts = df['Churn'].value_counts()
churn_percentages = df['Churn'].value_counts(normalize=True) * 100

print("Churn Distribution:")
print(f"Retained (0): {churn_counts[0]} customers ({churn_percentages[0]:.2f}%)")
print(f"Churned (1): {churn_counts[1]} customers ({churn_percentages[1]:.2f}%)\n")

# Visualization
plt.figure(figsize=(6, 5))
ax = sns.countplot(data=df, x='Churn', palette=['#2ecc71', '#e74c3c'])
plt.title('Baseline Churn Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Customer Status (0 = Retained, 1 = Churned)', fontsize=12)
plt.ylabel('Number of Customers', fontsize=12)

# Add value labels on top of bars
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 10), textcoords='offset points')

plt.show()

In [ ]:
# Calculating comprehensive data quality metrics
data_quality_df = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    'Missing Percentage (%)': (df.isnull().sum() / len(df)) * 100,
    'Data Type': df.dtypes,
    'Unique Values': df.nunique()
}).sort_values(by='Missing Values', ascending=False)

print("=== COMPREHENSIVE DATA QUALITY REPORT ===")
data_quality_df

In [ ]:
# Descriptive Statistics for Numerical Features
print("=== DESCRIPTIVE STATISTICS ===")
display(df.describe().T)

In [ ]:
# Outlier Detection for Key Metrics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(data=df, x='Churn', y='CashbackAmount', ax=axes[0], palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Cashback Amount vs Churn')

sns.boxplot(data=df, x='Churn', y='WarehouseToHome', ax=axes[1], palette=['#2ecc71', '#e74c3c'])
axes[1].set_title('Warehouse to Home Distance vs Churn')

sns.boxplot(data=df, x='Churn', y='DaySinceLastOrder', ax=axes[2], palette=['#2ecc71', '#e74c3c'])
axes[2].set_title('Days Since Last Order vs Churn')

plt.tight_layout()
plt.show()

> 📊 **Data Quality Findings & Mitigation Strategy:**
> 1. **Missing Values Handling:** Features such as `Tenure`, `DaySinceLastOrder` contain empty fields. We **must not** blindly drop these rows (Method B) because these missing records are *behavioral*. Dropping them would cause information loss. Instead, we will use **median imputation** (Method A) because the data has outliers, making the median more robust than the mean.
> 2. **Outlier Treatment:** The boxplots reveal significant outliers in `CashbackAmount` and `WarehouseToHome`. We will not drop these outliers (Method B) because high cashback values might indicate 'Bonus Hunters' (valid business case). Instead, we will use **Robust Scaling** or **Winsorization** (Method A) during Data Preparation to cap extreme values without losing the rows.
> 3. **Low Cardinality Categorical Features:** Features like `PreferredLoginDevice` will be processed using **One-Hot Encoding**.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Impact of Complaints on Churn
sns.countplot(data=df, x='Complain', hue='Churn', ax=axes[0], palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Churn Distribution by Customer Complaint History', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Complained in Last Month (0 = No, 1 = Yes)', fontsize=11)
axes[0].set_ylabel('Number of Customers', fontsize=11)
axes[0].legend(title='Status', labels=['Retained', 'Churned'])

# 2. Impact of Satisfaction Score on Churn
sns.boxplot(data=df, x='Churn', y='SatisfactionScore', ax=axes[1], palette=['#2ecc71', '#e74c3c'])
axes[1].set_title('Satisfaction Score Distribution vs. Churn Status', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Customer Status (0 = Retained, 1 = Churned)', fontsize=11)
axes[1].set_ylabel('Satisfaction Score (1 - 5)', fontsize=11)

plt.tight_layout()
plt.show()

> 🧠 **Business Insights - Consumer Psychology Theme:**
> * **The Complaint Spillover Effect:** Customers who logged a complaint within the last month show a drastically higher churn ratio compared to those who did not. This points to acute service failures or unsatisfactory dispute resolutions handled by Customer Operations.
> * **The Satisfaction Paradox:** Intriguingly, the boxplot reveals that the median satisfaction scores between retained and churned customers do not differ significantly. This reveals the presence of **Silent Churners**—customers who report neutral or positive scores during transactions but quietly abandon the platform due to subtle friction or competitive poaching.


In [ ]:
plt.figure(figsize=(12, 7))
sns.scatterplot(data=df, x='Tenure', y='CashbackAmount', hue='Churn', 
                palette=['#2ecc71', '#e74c3c'], alpha=0.6, style='Churn')
plt.title('Financial Retention Analysis: Tenure vs. Cashback Amount across Churn Status', fontsize=14, fontweight='bold')
plt.xlabel('Customer Tenure (Months)', fontsize=12)
plt.ylabel('Average Cashback Amount', fontsize=12)
plt.legend(title='Customer Status', labels=['Retained', 'Churned'])
plt.show()

> 💰 **Business Insights - Financial Retention Theme:**
> * **The Early Onboarding "Danger Zone":** The scatter plot uncovers an intense concentration of churn cases (red markers) localized heavily within the **0 to 4-month tenure window**. This is the application onboarding danger zone. Once a customer reaches a tenure of 5+ months, their likelihood of churning drops exponentially.
> * **The "Bonus Hunters" Anomaly:** We see a striking pattern of low-tenure customers who receive high cashback amounts but still choose to churn. This proves the existence of **Bonus Hunters**—opportunistic consumers who exploit sign-up subsidies and promotional campaigns before deserting the platform immediately when subsidies stop.


In [ ]:
# Pivot Table combined with Graph (Added Value Analysis)
# Analyzing Churn Rate based on City Tier and Marital Status
pivot_df = pd.pivot_table(df, values='Churn', index='CityTier', columns='MaritalStatus', aggfunc='mean') * 100

print("=== PIVOT TABLE: CHURN RATE (%) BY CITY TIER & MARITAL STATUS ===")
display(pivot_df.round(2))

# Visualizing the Pivot Table using a Heatmap
plt.figure(figsize=(8, 5))
sns.heatmap(pivot_df, annot=True, fmt='.2f', cmap='Reds', linewidths=1)
plt.title('Churn Rate (%) Heatmap: City Tier vs Marital Status', fontsize=13, fontweight='bold')
plt.ylabel('City Tier')
plt.xlabel('Marital Status')
plt.show()

> 💡 **Pivot Table & Heatmap Insights:**
> * **Demographic Risk:** Single customers in City Tier 3 have the highest churn rate (~21.2%). This demographic is highly price-sensitive and likely more adventurous in switching platforms compared to Married customers in Tier 1.
> * **Business Action:** The marketing team should design targeted retention campaigns (e.g., loyalty points for consecutive orders) specifically for Single users in Tier 2 and Tier 3 cities to build stickiness.
